# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")
# Print date published and citeAs if present
print(f"Date Published: {getattr(metadata, 'datePublished', None)}")
print(f"Cite As: {getattr(metadata, 'citeAs', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This will help in selecting which data to extract and analyze.

**Note:** All references to record sets and fields are done via their `@id` (identifier) fields for reproducibility and correctness.

In [ ]:
# List available record sets and enumerate their fields

print("Available Record Sets and their fields:")
record_sets = dataset.metadata.recordSets
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    print(f"- RecordSet @id: '{rs_id}', name: '{rs.get('name','')}'")
    fields = rs.get('fields', [])
    for f in fields:
        f_id = f['@id']
        f_name = f.get('name', '')
        f_dtype = f.get('dataType', '')
        print(f"    - Field @id: '{f_id}', name: '{f_name}', dataType: {f_dtype}")
print("\nList of record set @id's:")
print(record_set_ids)

# For demonstration, inspect the first record set's sample data
if len(record_set_ids) > 0:
    first_record_set = record_set_ids[0]
    print(f"\nSample records for record set '@id': {first_record_set}")
    for i, rec in enumerate(dataset.records(record_set=first_record_set)):
        print(json.dumps(rec, indent=2))
        if i >= 2:  # print only 3 records
            break

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis.

*Reference all record and field entities via their `@id` as explored above.*


In [ ]:
dataframes = {}
for record_set in record_set_ids:
    # Each record set yields records (dictionaries using field @id as keys)
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set])} rows for record set '{record_set}'")
    print(f"Columns (field @id): {dataframes[record_set].columns.tolist()}")
    print(dataframes[record_set].head(2))
    print("-"*60)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by categorical attributes.

For this demo, we will pick:
- The first record set for analysis (replace with another record set `@id` as preferred)
- A numeric field (`@id`) and a group field (`@id`) as observed above

*Update the variable values below to the appropriate field `@id`s as per the record set structure you found in the overview step.*


In [ ]:
# Edit these values based on the fields available in your chosen record set

# For demonstration we will use the first record_set
record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
df = dataframes[record_set_id]
# List the available columns (field @id)
print("All fields in selected record set:", df.columns.tolist())

# Example field @id and group field (replace as appropriate from the above listing)
# We'll attempt to find a likely numeric field and group field name
numeric_field_candidates = [col for col in df.columns if any(k in col.lower() for k in ['abundance','count','percent','mw','pi','number','coverage'])]
print("Candidate numeric field @id's:", numeric_field_candidates)
numeric_field = numeric_field_candidates[0] if len(numeric_field_candidates) > 0 else None
group_field_candidates = [col for col in df.columns if any(k in col.lower() for k in ['description','accession','type','sample'])]
group_field = group_field_candidates[0] if len(group_field_candidates) > 0 else None
print(f"Selected numeric_field: '{numeric_field}', group_field: '{group_field}'")

# Make sure field exists
if numeric_field is not None and numeric_field in df.columns:
    # Try to cast column to float if possible
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    # Filtering: keep values > threshold (e.g., 10)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with '{numeric_field}' > {threshold}: {len(filtered_df)} rows")
    print(filtered_df[[numeric_field]].head())
    # Normalization (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if available
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped (mean {numeric_field}) by '{group_field}':")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for demonstration. Please choose an appropriate field.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relation to the group field.

`Note:` The specific columns are referenced by their `@id` as per the Croissant schema.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None and group_field in df.columns:
        top_groups = df[group_field].value_counts().head(10).index
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset package using `mlcroissant`. By referencing all record sets and fields via their `@id`, we ensured reproducible and semantically precise access to the data schema. Analysts can now perform detailed downstream bioinformatics or statistical analyses on the protein abundance and peptide data present in the dataset.